# Task 3 — Dimensionality Reduction Techniques
**Machine Learning & Deep Learning Course Project**  
**Points: 10 pts**

---

## 1. Theoretical Background

### 1.1 Principal Component Analysis (PCA)

**Mathematical Formulation:**  
PCA finds a linear projection `W ∈ ℝᵈˣᵏ` that maximizes the variance of the projected data. The objective is:
```
maximize  Var(WᵀX) = Wᵀ Σ W
subject to WᵀW = I
```
where `Σ` is the sample covariance matrix `Σ = (1/(n-1)) XᵀX` (for mean-centered X).

**Algorithm:**
1. Center the data: `X̄ = X - μ`
2. Compute covariance matrix: `Σ = (1/(n-1)) X̄ᵀX̄`
3. Eigendecomposition: `Σ = V Λ Vᵀ` where columns of V are eigenvectors, Λ = diag(λ₁ ≥ λ₂ ≥ ... ≥ λd)
4. Select top k eigenvectors: `W = [v₁, v₂, ..., vₖ]`
5. Project: `Z = X̄ W` → reduced representation

**What is preserved vs lost:**
- ✅ Preserved: Global variance structure, pairwise Euclidean distances (approximately), linear correlations
- ❌ Lost: Non-linear relationships, local neighborhood structure, information in discarded components

**Complexity:**
- Covariance: O(n·d²)
- Eigendecomposition: O(d³) — bottleneck for high-d; use SVD: O(min(n,d)²·max(n,d))
- Scalability: Use `TruncatedSVD` (no centering needed) or `IncrementalPCA` for large datasets

---

### 1.2 t-SNE (t-distributed Stochastic Neighbor Embedding)

**Mathematical Formulation:**  
t-SNE converts high-dimensional pairwise similarities into probabilities and minimizes the KL divergence between the high-dimensional and low-dimensional distributions.

**High-dimensional similarity (Gaussian):**
```
pⱼ|ᵢ = exp(-||xᵢ - xⱼ||² / 2σᵢ²) / Σₖ≠ᵢ exp(-||xᵢ - xₖ||² / 2σᵢ²)
pᵢⱼ = (pⱼ|ᵢ + pᵢ|ⱼ) / 2n
```

**Low-dimensional similarity (Student-t with df=1):**
```
qᵢⱼ = (1 + ||yᵢ - yⱼ||²)⁻¹ / Σₖ≠ₗ (1 + ||yₖ - yₗ||²)⁻¹
```

**Objective (KL divergence):**
```
C = KL(P||Q) = Σᵢⱼ pᵢⱼ log(pᵢⱼ / qᵢⱼ)
```
Minimized via gradient descent: `∂C/∂yᵢ = 4 Σⱼ (pᵢⱼ - qᵢⱼ)(yᵢ - yⱼ)(1 + ||yᵢ - yⱼ||²)⁻¹`

The heavy-tailed t-distribution in low-D prevents the "crowding problem" — distant points are spread out more.

**What is preserved vs lost:**
- ✅ Preserved: Local neighborhood structure, cluster boundaries
- ❌ Lost: Global distances, interpretable axes, reproducibility (stochastic), cannot project new points

**Complexity:** O(n²) per iteration (or O(n log n) with Barnes-Hut approximation)  
**Scalability:** Impractical for n > 100K without approximation; apply PCA first to reduce to ~50 dims before t-SNE.

---

### 1.3 UMAP

UMAP (McInnes et al., 2018) is built on Riemannian geometry and algebraic topology. It constructs a fuzzy topological representation of the high-dimensional data and optimizes a low-dimensional equivalent. It preserves both local AND some global structure better than t-SNE, and is significantly faster: O(n^1.14) empirically.

### 1.4 LDA (Linear Discriminant Analysis)

LDA is a supervised dimensionality reduction technique. It maximizes the ratio of between-class scatter to within-class scatter:
```
J(W) = |Wᵀ Sᵦ W| / |Wᵀ Sᵥ W|
```
where Sᵦ = between-class scatter matrix, Sᵥ = within-class scatter matrix.  
Maximum dimensionality = min(d, C-1) where C = number of classes.

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, json
from pathlib import Path
SMOKE = os.getenv('ML_LAB_SMOKE') == '1'
Path('results').mkdir(exist_ok=True)
import warnings


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


try:
    import umap
    UMAP_AVAILABLE = True
    print("UMAP available ✓")
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP not installed. Install with: pip install umap-learn")
    print("Continuing without UMAP visualization.")

print("Imports successful ✓")

In [ ]:
# Visualizations use the training partition only.
from ml_lab.experiments import load_dataset, split_indices, make_pipeline
from sklearn.dummy import DummyClassifier

data_A = load_dataset("diabetes", quick=SMOKE)
data_B = load_dataset("digits", quick=SMOKE)
train_A, _, _ = split_indices(data_A.y)
train_B, _, _ = split_indices(data_B.y)
X_A, y_A, features_A = data_A.X[train_A], data_A.y[train_A], data_A.features
X_B, y_B, features_B = data_B.X[train_B], data_B.y[train_B], data_B.features
preprocess_A = make_pipeline(DummyClassifier(), zero_missing=True)[:-1]
preprocess_B = make_pipeline(DummyClassifier())[:-1]
X_A_scaled = preprocess_A.fit_transform(X_A)
X_B_scaled = preprocess_B.fit_transform(X_B)
print("Training-only exploratory shapes:", X_A_scaled.shape, X_B_scaled.shape)


---
## 2. Practical Application
### 2.1 PCA — Explained Variance Ratio (Elbow Plot)

In [ ]:
def plot_pca_variance(X_scaled, title, ax1, ax2, color='#3498DB'):
    """Plot individual and cumulative explained variance ratios."""
    pca_full = PCA()
    pca_full.fit(X_scaled)
    evr = pca_full.explained_variance_ratio_
    cumulative = np.cumsum(evr)
    n_components = len(evr)
    
    # Find k for 95% variance
    k_95 = np.argmax(cumulative >= 0.95) + 1
    
    # Individual variance
    ax1.bar(range(1, n_components+1), evr * 100, color=color, alpha=0.7, edgecolor='black', lw=0.5)
    ax1.set_title(f'{title}\nIndividual Explained Variance', fontsize=11)
    ax1.set_xlabel('Principal Component')
    ax1.set_ylabel('Explained Variance (%)')
    ax1.grid(True, axis='y', alpha=0.3)
    
    # Cumulative variance
    ax2.plot(range(1, n_components+1), cumulative * 100,
             color=color, linewidth=2.5, marker='o', markersize=5)
    ax2.axhline(95, color='red', linestyle='--', linewidth=1.5, label='95% threshold')
    ax2.axvline(k_95, color='green', linestyle=':', linewidth=2,
                label=f'k={k_95} components')
    ax2.set_title(f'{title}\nCumulative Explained Variance', fontsize=11)
    ax2.set_xlabel('Number of Components')
    ax2.set_ylabel('Cumulative Variance (%)')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 102)
    
    print(f"{title}: {k_95} components retain 95% of variance")
    return k_95

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('PCA — Explained Variance Analysis', fontsize=14, fontweight='bold')

k95_A = plot_pca_variance(X_A_scaled, 'Diabetes (8 features)',
                           axes[0][0], axes[0][1], '#E74C3C')
k95_B = plot_pca_variance(X_B_scaled, 'Digits (64 features)',
                           axes[1][0], axes[1][1], '#3498DB')

plt.tight_layout()
plt.savefig('task3_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 2D Projections — PCA, t-SNE, UMAP

In [ ]:
def plot_2d_projections(X_scaled, y, title, class_labels=None):
    """
    Apply PCA, t-SNE, and UMAP and visualize 2D projections.
    """
    print(f"Computing projections for {title}...")
    
    # PCA
    pca2 = PCA(n_components=2, random_state=42)
    X_pca = pca2.fit_transform(X_scaled)
    print(f"  PCA done. PC1={pca2.explained_variance_ratio_[0]:.3f}, PC2={pca2.explained_variance_ratio_[1]:.3f}")
    
    # t-SNE
    tsne = TSNE(n_components=2, perplexity=30, max_iter=250 if SMOKE else 1000, random_state=42, learning_rate='auto', init='pca')
    X_tsne = tsne.fit_transform(X_scaled)
    print(f"  t-SNE done.")
    
    # UMAP
    if UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=42)
        X_umap = reducer.fit_transform(X_scaled)
        print(f"  UMAP done.")
        n_plots = 4
    else:
        X_umap = None
        n_plots = 3
    
    # LDA
    n_components_lda = min(len(np.unique(y)) - 1, X_scaled.shape[1], 2)
    lda = LinearDiscriminantAnalysis(n_components=n_components_lda)
    X_lda = lda.fit_transform(X_scaled, y)
    if X_lda.shape[1] == 1:
        X_lda = np.column_stack([X_lda, np.zeros(len(X_lda))])
    print(f"  LDA done. Components: {n_components_lda}")
    
    classes = np.unique(y)
    palette = plt.cm.tab10(np.linspace(0, 1, len(classes)))
    if class_labels is None:
        class_labels = [f'Class {c}' for c in classes]
    
    fig, axes = plt.subplots(1, n_plots, figsize=(5.5 * n_plots, 5))
    fig.suptitle(f'2D Projections — {title}', fontsize=13, fontweight='bold')
    
    plot_data = [
        (X_pca, f'PCA\n(PC1={pca2.explained_variance_ratio_[0]:.2%}, PC2={pca2.explained_variance_ratio_[1]:.2%})'),
        (X_tsne, 't-SNE (perplexity=30)'),
        (X_lda, 'LDA'),
    ]
    if X_umap is not None:
        plot_data.append((X_umap, 'UMAP'))
    
    for ax, (X_2d, method_title) in zip(axes, plot_data):
        for i, cls in enumerate(classes):
            mask = y == cls
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                       color=palette[i % len(palette)],
                       label=class_labels[i], alpha=0.65, s=25, edgecolors='k', linewidths=0.2)
        ax.set_title(method_title, fontsize=11)
        ax.legend(fontsize=9, loc='best')
        ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(f'task3_projections_{title.lower().replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return X_pca, X_tsne, X_lda

X_pca_A, X_tsne_A, X_lda_A = plot_2d_projections(
    X_A_scaled, y_A, 'Diabetes', ['No Diabetes', 'Diabetes']
)
digit_labels = [f'Digit {i}' for i in range(10)]
X_pca_B, X_tsne_B, X_lda_B = plot_2d_projections(
    X_B_scaled, y_B, 'Digits', digit_labels
)

### 2.3 PCA-Reduced Classification Performance

In [ ]:
# Each fold fits its own imputer, scaler and PCA from raw training rows.
def pca_classification_comparison(X_raw, y, dataset_name, zero_missing=False):
    cv = StratifiedKFold(n_splits=3 if SMOKE else 5, shuffle=True, random_state=42)
    rows = []
    for label, components in [("Full features", None), ("PCA 95% variance", 0.95), ("PCA 2D", 2)]:
        pipeline = make_pipeline(
            RandomForestClassifier(n_estimators=20 if SMOKE else 100,
                                   random_state=42, n_jobs=1),
            zero_missing=zero_missing, pca=components)
        scores = cross_val_score(pipeline, X_raw, y, cv=cv, scoring="accuracy",
                                 n_jobs=1, error_score="raise")
        rows.append({"transform": label, "cv_accuracy_mean": float(scores.mean()),
                     "cv_accuracy_std": float(scores.std())})
    report = {"schema_version": 1, "kind": "pca", "dataset": dataset_name,
              "mode": "smoke" if SMOKE else "full", "results": rows}
    Path(f"results/task3_{dataset_name}.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8")
    frame = pd.DataFrame(rows)
    print(dataset_name, frame.to_string(index=False))
    ax = frame.plot.bar(x="transform", y="cv_accuracy_mean", yerr="cv_accuracy_std",
                        legend=False, ylim=(0, 1), title=f"{dataset_name}: training-only CV accuracy")
    ax.figure.tight_layout()
    ax.figure.savefig(f"results/task3_pca_{dataset_name}.png", dpi=120)
    plt.show()
    return frame

res_A = pca_classification_comparison(X_A, y_A, "diabetes", zero_missing=True)
res_B = pca_classification_comparison(X_B, y_B, "digits")


## Interpretation

Exploratory projections are fitted on training rows only. LDA uses training labels;
visual separation is not evidence of held-out performance. t-SNE and UMAP plots do not
establish generalization accuracy.

The classifier comparison refits imputation, scaling and PCA within each CV training
fold. The 95% variance threshold is fixed in advance; its component count is learned
independently inside each fold. These training-partition CV accuracy scores are
separate from the final test accuracy in Task 2 and must not be merged as the same metric.

Use the generated results to discuss how compression changes performance. No fixed
accuracy or universal PCA benefit is claimed.